<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W5D5_Mini_Projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# DAILY CHALLENGE - CATS VS DOGS CLASSIFICATION
# COMPLETE SOLUTION IN ONE CELL
# ==========================================================

# =========================
# Imports
# =========================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# ==========================================================
# 1. DATA REPORT
# ==========================================================

print("===== CLASS INDICES =====")
print(train_flow.class_indices)

labels = train_flow.labels

unique, counts = np.unique(labels, return_counts=True)

print("\n===== CLASS COUNTS =====")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

if abs(counts[0] - counts[1]) < 0.10 * len(labels):
    print("\nDataset is approximately balanced.")
else:
    print("\nDataset is imbalanced.")

# ==========================================================
# 2. IMAGE GRID
# ==========================================================

images, batch_labels = next(train_flow)

plt.figure(figsize=(12,8))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    label_name = "dog" if batch_labels[i] == 1 else "cat"

    plt.title(label_name)

    plt.axis("off")

plt.tight_layout()
plt.show()

print("""
Visual cues:
- Ear shape
- Face shape
- Fur texture
- Body proportions
- Background context
""")

# ==========================================================
# 3. CNN ARCHITECTURE
# ==========================================================

print("""
CNN Architecture:
- 3 convolution blocks
- Filters: 32 -> 64 -> 128
- MaxPooling after each block
- Dropout for regularization
- Dense hidden layer
- Sigmoid output for binary classification
""")

model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    ),

    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),
    MaxPooling2D(),

    Dropout(0.3),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

# ==========================================================
# 4. OPTIMIZATION SETUP
# ==========================================================

learning_rate = 0.001

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2
)

# ==========================================================
# 5. CLASS WEIGHTS
# ==========================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weight_dict = {
    i: w
    for i, w in enumerate(class_weights)
}

print("\nClass Weights:")
print(class_weight_dict)

# ==========================================================
# 6. TRAIN MODEL WITH AUGMENTATION
# ==========================================================

history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=20,
    callbacks=[
        early_stop,
        reduce_lr
    ],
    class_weight=class_weight_dict,
    verbose=1
)

# ==========================================================
# 7. TRAINING CURVES
# ==========================================================

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)

plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])

plt.title("Accuracy")
plt.legend(["Train","Validation"])

plt.subplot(1,2,2)

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])

plt.title("Loss")
plt.legend(["Train","Validation"])

plt.show()

# ==========================================================
# 8. VALIDATION EVALUATION
# ==========================================================

val_loss, val_acc = model.evaluate(
    val_flow,
    verbose=0
)

print("\n===== VALIDATION RESULTS =====")
print("Validation Loss :", round(val_loss,4))
print("Validation Accuracy :", round(val_acc,4))

# ==========================================================
# 9. CONFUSION MATRIX
# ==========================================================

probabilities = model.predict(val_flow)

predictions = (
    probabilities > 0.5
).astype(int)

true_labels = val_flow.classes

cm = confusion_matrix(
    true_labels,
    predictions
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

print(
    classification_report(
        true_labels,
        predictions,
        target_names=["cat","dog"]
    )
)

# ==========================================================
# 10. TEST INFERENCE
# ==========================================================

test_probabilities = model.predict(
    test_flow
)

threshold = 0.5

test_labels = np.where(
    test_probabilities > threshold,
    "dog",
    "cat"
)

submission = pd.DataFrame({
    "filepath": test_flow.filepaths,
    "prob_dog": test_probabilities.flatten(),
    "pred_label": test_labels.flatten()
})

submission.to_csv(
    "test_predictions.csv",
    index=False
)

print("\nPredictions saved:")
print("test_predictions.csv")

# ==========================================================
# 11. BASELINE MODEL (NO AUGMENTATION)
# ==========================================================

baseline_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

baseline_flow = baseline_gen.flow_from_dataframe(
    df_tr,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    batch_size=batch_size,
    shuffle=True
)

baseline_model = tf.keras.models.clone_model(model)

baseline_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

baseline_history = baseline_model.fit(
    baseline_flow,
    validation_data=val_flow,
    epochs=5,
    verbose=1
)

baseline_acc = max(
    baseline_history.history["val_accuracy"]
)

augmented_acc = max(
    history.history["val_accuracy"]
)

print("\n===== COMPARISON =====")
print("Baseline Validation Accuracy :", baseline_acc)
print("Augmented Validation Accuracy :", augmented_acc)

# ==========================================================
# 12. SAVE MODEL
# ==========================================================

model.save("cats_dogs_model.h5")

config = {
    "img_height": IMG_HEIGHT,
    "img_width": IMG_WIDTH,
    "batch_size": batch_size,
    "optimizer": "Adam",
    "learning_rate": learning_rate,
    "loss": "binary_crossentropy"
}

with open(
    "training_config.json",
    "w"
) as f:
    json.dump(
        config,
        f,
        indent=4
    )

print("\nSaved:")
print("- cats_dogs_model.h5")
print("- training_config.json")

# ==========================================================
# 13. EXTENSION PROPOSAL
# ==========================================================

print("""
Recommended Extension:
Transfer Learning with MobileNetV2.

Benefits:
- Faster convergence
- Better feature extraction
- Higher validation accuracy
- Less training data required
""")